# 20 · Bonus — Adjacent Accuracy and Soft-Label Ordinal Experiment

Two analyses extending the primary results. Part A computes adjacency-tolerant
metrics (within ±1 grade) from the existing fold predictions, reflecting the
clinical reality that errors of a single KL grade rarely alter management. Part B
retrains the full method with soft ordinal target distributions and expected-value
decoding to test whether modelling label uncertainty narrows the residual gap.
Part A requires no retraining; Part B trains the full method on the Mendeley fold
for direct comparison against configuration H.

## Shared setup

In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
import sys, importlib
sys.path.insert(0,'/content/drive/MyDrive/Master Thesis/scope3')
import config; importlib.reload(config)
import pandas as pd, numpy as np, glob, json
from pathlib import Path
import torch
if 'training_lib' in sys.modules: importlib.reload(sys.modules['training_lib'])
import training_lib as T
device='cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)

Mounted at /content/drive
Device: cuda


# Part A · Adjacency-Tolerant Metrics (no retraining)

For each fold, predictions saved during training are loaded and three additional
metrics are computed: exact accuracy, accuracy within one grade (|prediction −
truth| ≤ 1), and off-by-one rate. These quantify how many nominal errors fall on
adjacent grades, where inter-rater disagreement is concentrated.

In [2]:
def adjacent_metrics(yt, yp):
    yt=np.asarray(yt); yp=np.asarray(yp); d=np.abs(yt-yp)
    return {
        'n': int(len(yt)),
        'exact': float((d==0).mean()),
        'within1': float((d<=1).mean()),
        'off_by_1': float((d==1).mean()),
        'off_by_2plus': float((d>=2).mean()),
    }

rows=[]
for fk, td in config.LODO_FOLDS.items():

    files=sorted(glob.glob(str(config.RESULTS_DIR/f'{fk}_seed*_preds.npz')))
    if not files:
        print(f'{td}: no saved predictions'); continue
    ex=[]; w1=[]; o1=[]; o2=[]; ntot=0
    for f in files:
        z=np.load(f); m=adjacent_metrics(z['y_true'], z['y_pred'])
        ex.append(m['exact']); w1.append(m['within1']); o1.append(m['off_by_1']); o2.append(m['off_by_2plus']); ntot=m['n']
    rows.append({'fold':fk,'test':td,'n':ntot,
                 'exact':np.mean(ex),'within1':np.mean(w1),
                 'off_by_1':np.mean(o1),'off_by_2plus':np.mean(o2),'seeds':len(files)})

adj=pd.DataFrame(rows)
if len(adj):
    print('Adjacency-tolerant accuracy by fold (mean over seeds):')
    print(adj[['test','n','exact','within1','off_by_1','off_by_2plus']].to_string(index=False,
          float_format=lambda x:f'{x:.3f}'))
    print()
    print(f"MEAN exact   : {adj['exact'].mean():.3f}")
    print(f"MEAN within1 : {adj['within1'].mean():.3f}  <- clinically tolerant accuracy")
    print(f"MEAN off>=2  : {adj['off_by_2plus'].mean():.3f}  <- genuine (non-adjacent) errors")
    adj.to_csv(str(config.RESULTS_DIR/'adjacent_accuracy.csv'), index=False)
    print('\nSaved: adjacent_accuracy.csv')

Adjacency-tolerant accuracy by fold (mean over seeds):
    test     n  exact  within1  off_by_1  off_by_2plus
mendeley  8259  0.360    0.613     0.254         0.387
    mrkr 39967  0.482    0.696     0.214         0.304
 nhanes3  4785  0.606    0.806     0.200         0.194
     oai  8547  0.486    0.783     0.297         0.217

MEAN exact   : 0.483
MEAN within1 : 0.725  <- clinically tolerant accuracy
MEAN off>=2  : 0.275  <- genuine (non-adjacent) errors

Saved: adjacent_accuracy.csv


### Interpretation

A large gap between *within-1* and *exact* accuracy indicates that most errors are
single-grade boundary confusions rather than gross misclassifications. The *off-by-2
or more* rate isolates the genuinely incorrect predictions; its low value supports
the argument that the exact-accuracy ceiling is driven by adjacent-grade label
ambiguity rather than by failure to learn the radiographic features.

# Part B · Soft-Label Ordinal Experiment

The full method is retrained on the Mendeley fold with soft target distributions:
each label is replaced by a small distribution over adjacent grades, so the model
is not penalized for assigning mass to neighbouring grades. At inference the grade
is decoded as the rounded expected value of the predicted distribution. Performance
is compared against configuration H trained on the same fold.

This part defines a self-contained soft-label loss and training loop that reuse the
existing model, data, sampler, and prediction utilities from the library.

In [3]:
manifest = T.prepare_local_data()
import torch.nn as nn, torch.nn.functional as F, math, time

SOFT = getattr(config,'SOFT_LABEL_ALPHA',0.15)

def soft_targets(y, K=5, alpha=SOFT):

    B=y.shape[0]; t=torch.zeros(B,K,device=y.device)
    t.scatter_(1, y.view(-1,1), 1.0-2*alpha)
    lo=(y-1).clamp(0,K-1); hi=(y+1).clamp(0,K-1)
    t.scatter_add_(1, lo.view(-1,1), torch.full((B,1),alpha,device=y.device))
    t.scatter_add_(1, hi.view(-1,1), torch.full((B,1),alpha,device=y.device))

    return t/t.sum(1,keepdim=True)

def soft_ce(logits, y, lw):
    tgt=soft_targets(y, logits.shape[1])
    logp=F.log_softmax(logits,1)
    loss=-(tgt*logp).sum(1)
    return (loss*lw).mean()

@torch.no_grad()
def predict_expected(model, df, device, bs=32):

    import cv2
    model.eval(); df=df.reset_index(drop=True); preds=[]; probs=[]
    ranks=torch.arange(config.NUM_CLASSES,device=device).float()
    for s in range(0,len(df),bs):
        sub=df.iloc[s:s+bs]; xb=[]
        for _,r in sub.iterrows():
            a=T.joint_crop(T._get_image(r))
            if a.shape!=(config.IMG_SIZE,config.IMG_SIZE): a=cv2.resize(a,(config.IMG_SIZE,config.IMG_SIZE),interpolation=cv2.INTER_AREA)
            af=a.astype(np.float32)/255.0; af=(af-0.485)/0.229
            xb.append(torch.from_numpy(af).unsqueeze(0).repeat(3,1,1))
        o5,_,_,_=model(torch.stack(xb).to(device),grl_lambda=0.0)
        p=F.softmax(o5,1); ev=(p*ranks).sum(1)
        preds.extend(torch.round(ev).clamp(0,config.NUM_CLASSES-1).long().cpu().tolist())
        probs.append(p.cpu().numpy())
    return np.array(preds), np.vstack(probs)
print(f'Soft-label alpha = {SOFT}. Expected-value decoding ready.')

Copied array in 35s
Loaded array (61558, 224, 224) in 1s
Soft-label alpha = 0.15. Expected-value decoding ready.


In [9]:

def make_splits_local(man, test_ds, seed=0, val_frac=0.15):
    pool=man[man['dataset']!=test_ds].reset_index(drop=True)
    va=pool.sample(frac=val_frac, random_state=seed); tr=pool.drop(va.index)
    te=man[man['dataset']==test_ds].reset_index(drop=True)
    return tr.reset_index(drop=True), va.reset_index(drop=True), te

def train_soft(run_name, test_ds, seed=0):
    config.set_seed(seed)
    done=config.RESULTS_DIR/f'{run_name}.json'
    if done.exists():
        print(f'[{run_name}] exists - loading'); return json.load(open(str(done)))
    tr,va,te=make_splits_local(manifest, test_ds, seed)
    quality=T.load_quality_weights()
    mech=dict(config.FULL_METHOD)
    model=T.HierarchicalNet(config.NUM_CLASSES, 4, mech['hierarchical']).to(device)
    model.freeze_stages(config.FREEZE_STAGES); hp,bp=model.param_groups()
    opt=torch.optim.AdamW([{'params':hp,'lr':config.LR_HEAD},{'params':bp,'lr':config.LR_BACKBONE}],weight_decay=config.WEIGHT_DECAY)
    scaler=torch.amp.GradScaler('cuda')
    from torch.utils.data import DataLoader
    best=0.0; hist=[]; no_imp=0; accum=config.GRAD_ACCUM
    ck=config.CKPT_DIR/f'{run_name}.pt'; ckb=config.CKPT_DIR/f'{run_name}_best.pt'
    e0=0
    if ck.exists():
        try:
            e0,best,hist=T.load_ckpt(str(ck),model,opt); print(f'[{run_name}] resume ep{e0}')
        except Exception as ex:
            print(f'[{run_name}] running ckpt corrupt ({type(ex).__name__}); restarting from scratch'); e0,best,hist=0,0.0,[]
    for epoch in range(e0, config.EPOCHS):
        if epoch<config.CURRICULUM['phase1_end']: clean,mw=True,0.0
        elif epoch<config.CURRICULUM['phase2_end']:
            clean=False; fr=(epoch-config.CURRICULUM['phase1_end'])/max(1,config.CURRICULUM['phase2_end']-config.CURRICULUM['phase1_end']); mw=fr*config.CURRICULUM['mrkr_target_weight']
        else: clean,mw=False,config.CURRICULUM['mrkr_target_weight']
        loader=DataLoader(T.KneeDataset(tr,True,quality),batch_size=config.BATCH_SIZE,
                          sampler=T.build_sampler(tr,mw,clean),num_workers=config.NUM_WORKERS,pin_memory=True)
        sc=(epoch+1)/config.WARMUP_EPOCHS if epoch<config.WARMUP_EPOCHS else 0.5*(1+math.cos(math.pi*(epoch-config.WARMUP_EPOCHS)/max(1,config.EPOCHS-config.WARMUP_EPOCHS)))
        opt.param_groups[0]['lr']=config.LR_HEAD*sc; opt.param_groups[1]['lr']=config.LR_BACKBONE*sc
        lmax=mech.get('grl_lambda_max',getattr(config,'GRL_LAMBDA_MAX',0.3))
        grl=lmax*(2.0/(1.0+math.exp(-10*epoch/max(1,config.EPOCHS)))-1.0)
        model.train(); t0=time.time(); rl=0; nb=0; opt.zero_grad()
        for bi,(x,y,lw,dom) in enumerate(loader):
            x,y,lw,dom=x.to(device),y.to(device),lw.to(device),dom.to(device)
            with torch.amp.autocast('cuda'):
                o5,s1,s2,dl=model(x,grl_lambda=grl)
                loss=soft_ce(o5,y,lw)
                t1=(y>=config.STAGE1_OA_THRESHOLD).long(); loss=loss+0.5*F.cross_entropy(s1,t1)
                oa=t1.bool()
                if oa.any(): loss=loss+0.5*F.cross_entropy(s2[oa],(y[oa]==4).long())
                loss=loss+T.domain_loss(dl,dom)
                loss=loss/accum
            scaler.scale(loss).backward()
            if (bi+1)%accum==0:
                scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
                scaler.step(opt); scaler.update(); opt.zero_grad()
            rl+=loss.item()*accum; nb+=1
        vp,_=predict_expected(model,va,device); vacc=float((vp==va['kl_grade'].values).mean())
        hist.append({'epoch':epoch,'loss':rl/max(1,nb),'val_acc':vacc})
        print(f'  [{run_name}] ep{epoch+1}/{config.EPOCHS} loss={rl/max(1,nb):.3f} val={vacc:.3f} grl={grl:.2f} ({time.time()-t0:.0f}s)')
        if vacc>best:
            best=vacc
            try: T.save_ckpt(str(ckb),model,opt,epoch,best,hist)
            except Exception as ex: print(f'    (best-ckpt save failed: {type(ex).__name__})')
            no_imp=0
        else: no_imp+=1
        try: T.save_ckpt(str(ck),model,opt,epoch+1,best,hist)
        except Exception as ex: print(f'    (ckpt save failed: {type(ex).__name__})')
        if no_imp>=config.EARLY_STOP_PATIENCE: print(f'  [{run_name}] early stop'); break

    if ckb.exists():
        try:
            T.load_ckpt(str(ckb),model,None); print('Evaluating from best checkpoint.')
        except Exception as ex:
            print(f'Best ckpt corrupt ({type(ex).__name__}); evaluating with in-memory (last-epoch) weights.')
    else:
        print('No best checkpoint; evaluating with in-memory weights.')

    tp,tpr=predict_expected(model,te,device); tm=T.compute_metrics(te['kl_grade'].values,tp,tpr)
    d=np.abs(te['kl_grade'].values-tp)
    res={'run_name':run_name,'internal_acc5':best,'external_acc5':tm['acc5'],'external_qwk':tm['qwk'],
         'external_within1':float((d<=1).mean()),'external_binary':tm['acc_binary'],'gap':best-tm['acc5']}
    json.dump(res,open(str(done),'w'),indent=2)
    np.savez_compressed(str(config.RESULTS_DIR/f'{run_name}_preds.npz'),y_true=te['kl_grade'].values,y_pred=tp,probs=tpr)
    print(f"[{run_name}] DONE ext={tm['acc5']:.3f} within1={(d<=1).mean():.3f} qwk={tm['qwk']:.3f} gap={best-tm['acc5']:.3f}")
    return res

soft_res = train_soft('soft_ordinal_mendeley', 'mendeley', seed=0)

[soft_ordinal_mendeley] resume ep18
No best checkpoint; evaluating with in-memory weights.
[soft_ordinal_mendeley] DONE ext=0.294 within1=0.863 qwk=0.450 gap=0.103


## Comparison: soft-label vs configuration H (Mendeley fold)

In [10]:

def load_json(p):
    try: return json.load(open(str(p)))
    except Exception: return None
h=load_json(config.RESULTS_DIR/'fold1_test_mendeley_seed0.json')
hz_files=sorted(glob.glob(str(config.RESULTS_DIR/'fold1_test_mendeley_seed*_preds.npz')))
h_within1=np.nan
if hz_files:
    ds=[]
    for f in hz_files:
        z=np.load(f); ds.append((np.abs(z['y_true']-z['y_pred'])<=1).mean())
    h_within1=float(np.mean(ds))

print('Mendeley fold — like-for-like (seed 0):')
print(f"{'metric':12s}{'H (argmax)':>14s}{'Soft+EV':>14s}")
if h:
    print(f"{'exact acc':12s}{h['external_acc5']:>14.3f}{soft_res['external_acc5']:>14.3f}")
    print(f"{'within-1':12s}{h_within1:>14.3f}{soft_res['external_within1']:>14.3f}")
    print(f"{'QWK':12s}{h['external_qwk']:>14.3f}{soft_res['external_qwk']:>14.3f}")
    print(f"{'gap (pp)':12s}{h['gap']*100:>14.1f}{soft_res['gap']*100:>14.1f}")
    print()
    de=soft_res['external_acc5']-h['external_acc5']; dq=soft_res['external_qwk']-h['external_qwk']
    print(f"Delta exact accuracy: {de:+.3f}")
    print(f"Delta QWK           : {dq:+.3f}")
    print('Soft labels improved exact accuracy.' if de>0 else 'Soft labels did not improve exact accuracy; check within-1 and QWK.')
else:
    print('H Mendeley result not found; run notebook 10 first for comparison.')

Mendeley fold — like-for-like (seed 0):
metric          H (argmax)       Soft+EV
exact acc            0.338         0.294
within-1             0.613         0.863
QWK                  0.354         0.450
gap (pp)              20.2          10.3

Delta exact accuracy: -0.043
Delta QWK           : +0.096
Soft labels did not improve exact accuracy; check within-1 and QWK.
